# 1. Load Data

In [38]:
import os 
print(os.getcwd())


/Users/raresolteanu/Desktop/WC-2026


In [39]:
os.chdir("/Users/raresolteanu/Desktop/WC-2026")
print(os.getcwd())

/Users/raresolteanu/Desktop/WC-2026


In [40]:
import pandas as pd
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    log_loss,
    classification_report,
    confusion_matrix
)

full_df = pd.read_csv("datasets/wc_2026_training_table_clean_with_spi_and_odds.csv")
odds_df = pd.read_csv("datasets/wc_2026_odds_experiment_model_table.csv")
schedule_2026 = pd.read_csv("datasets/WC 2026 match schedule advanced.csv")

for df in [full_df, odds_df, schedule_2026]:
    df["date"] = pd.to_datetime(df["date"])

print(full_df.shape)
print(odds_df.shape)
print(schedule_2026.shape)

(30696, 140)
(1080, 140)
(104, 75)


# 2. Define Features

In [41]:
baseline_features = [
    "elo_diff",
    "fifa_rank_diff",
    "fifa_points_zscore_diff",
    "form_score_5_diff",
    "form_score_10_diff",
    "form_trend_diff",
    "attack_diff",
    "defense_diff",
    "rest_days_diff",
    "neutral",
    "is_home_country",
    "is_home_confed",
    "is_away_confed",
    "missing_fifa_rank",
    "missing_fifa_points_zscore",
]

h2h_streak_features = [
    "h2h_matches_played",
    "h2h_wins_diff",
    "h2h_draws",
    "h2h_goals_diff",
    "h2h_avg_goals_diff",
    "home_win_streak",
    "home_unbeaten_streak",
    "away_win_streak",
    "away_unbeaten_streak",
    "win_streak_diff",
    "unbeaten_streak_diff",
]

odds_features = [
    "market_home_prob_norm",
    "market_draw_prob_norm",
    "market_away_prob_norm",
    "odds_prob_diff",
    "market_favorite_prob",
    "market_draw_prob",
]

full_features = baseline_features + h2h_streak_features

# 3. Prepare Data

In [42]:
for df in [full_df, odds_df, schedule_2026]:
    df["missing_fifa_rank"] = df["fifa_rank_diff"].isna().astype(int)
    df["missing_fifa_points_zscore"] = df["fifa_points_zscore_diff"].isna().astype(int)

    df["fifa_rank_diff"] = df["fifa_rank_diff"].fillna(0)
    df["fifa_points_zscore_diff"] = df["fifa_points_zscore_diff"].fillna(0)

In [43]:
def prepare_model_table(df, features):
    df = df.copy()

    for col in features:
        if col not in df.columns:
            raise ValueError(f"Missing feature: {col}")

    df[features] = df[features].replace([np.inf, -np.inf], np.nan)
    df[features] = df[features].fillna(0)

    return df


full_model = prepare_model_table(full_df, full_features)
odds_model = prepare_model_table(odds_df, odds_features)
schedule_model = prepare_model_table(schedule_2026, full_features + odds_features)

# 4. Time Splits

In [44]:
full_trainable = full_model[
    (full_model["date"] < "2026-01-01") &
    (full_model["result"].notna())
].copy()

odds_trainable = odds_model[
    (odds_model["date"] < "2026-01-01") &
    (odds_model["result"].notna())
].copy()

full_train = full_trainable[full_trainable["date"] < "2019-01-01"].copy()

odds_train = odds_trainable[odds_trainable["date"] < "2024-01-01"].copy()

odds_val = odds_trainable[
    (odds_trainable["date"] >= "2024-01-01") &
    (odds_trainable["date"] < "2025-01-01")
].copy()

odds_test = odds_trainable[
    (odds_trainable["date"] >= "2025-01-01") &
    (odds_trainable["date"] < "2026-01-01")
].copy()

# 5. Train Base Models

In [45]:
full_base_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=42
    ))
])

odds_base_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=42
    ))
])

full_base_model.fit(full_train[full_features], full_train["result"])
odds_base_model.fit(odds_train[odds_features], odds_train["result"])

,steps,"[('scaler', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,copy,True
,with_mean,True
,with_std,True
,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0


# 6. Build Stacking Features

In [46]:
def make_stacking_features(df):
    full_probs = full_base_model.predict_proba(df[full_features])
    odds_probs = odds_base_model.predict_proba(df[odds_features])

    full_prob_df = pd.DataFrame(
        full_probs,
        columns=[f"full_prob_{c}" for c in full_base_model.classes_],
        index=df.index
    )

    odds_prob_df = pd.DataFrame(
        odds_probs,
        columns=[f"odds_prob_{c}" for c in odds_base_model.classes_],
        index=df.index
    )

    return pd.concat([full_prob_df, odds_prob_df], axis=1)


X_stack_val = make_stacking_features(odds_val)
y_stack_val = odds_val["result"]

X_stack_test = make_stacking_features(odds_test)
y_stack_test = odds_test["result"]

# 7. Train Stacked LR Model

In [47]:
stack_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

stack_model.fit(X_stack_val, y_stack_val)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,'balanced'
,random_state,42
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


# 8. Check Test Performance

In [48]:
stack_test_preds = stack_model.predict(X_stack_test)
stack_test_probs = stack_model.predict_proba(X_stack_test)

print("Stacked LR - Full + Odds - Test")
print("Accuracy:", round(accuracy_score(y_stack_test, stack_test_preds), 4))
print("Balanced accuracy:", round(balanced_accuracy_score(y_stack_test, stack_test_preds), 4))
print("Macro F1:", round(f1_score(y_stack_test, stack_test_preds, average="macro"), 4))
print("Weighted F1:", round(f1_score(y_stack_test, stack_test_preds, average="weighted"), 4))
print("Log loss:", round(log_loss(y_stack_test, stack_test_probs, labels=stack_model.classes_), 4))

print("\nClassification report:")
print(classification_report(y_stack_test, stack_test_preds))

print("Confusion matrix:")
print(confusion_matrix(y_stack_test, stack_test_preds, labels=stack_model.classes_))

Stacked LR - Full + Odds - Test
Accuracy: 0.6429
Balanced accuracy: 0.6181
Macro F1: 0.6148
Weighted F1: 0.6649
Log loss: 0.7848

Classification report:
              precision    recall  f1-score   support

    away_win       0.73      0.65      0.69       154
        draw       0.32      0.51      0.39       100
    home_win       0.85      0.69      0.77       236

    accuracy                           0.64       490
   macro avg       0.63      0.62      0.61       490
weighted avg       0.71      0.64      0.66       490

Confusion matrix:
[[100  51   3]
 [ 24  51  25]
 [ 13  59 164]]


# 9. Predict WC 2026 Matches

In [49]:
X_schedule_stack = make_stacking_features(schedule_model)

schedule_probs = stack_model.predict_proba(X_schedule_stack)
schedule_preds = stack_model.predict(X_schedule_stack)

predictions_2026 = schedule_2026.copy()

for i, class_name in enumerate(stack_model.classes_):
    predictions_2026[f"prob_{class_name}"] = schedule_probs[:, i]

for col in ["prob_home_win", "prob_draw", "prob_away_win"]:
    if col not in predictions_2026.columns:
        predictions_2026[col] = 0

predictions_2026["predicted_result"] = schedule_preds
predictions_2026["prediction_confidence"] = schedule_probs.max(axis=1)

predictions_2026 = predictions_2026.sort_values("date").reset_index(drop=True)

display(
    predictions_2026[
        [
            "date",
            "home_team",
            "away_team",
            "group",
            "predicted_result",
            "prob_home_win",
            "prob_draw",
            "prob_away_win",
            "prediction_confidence",
        ]
    ].head(30)
)

,date,home_team,away_team,group,predicted_result,prob_home_win,prob_draw,prob_away_win,prediction_confidence
0,2026-06-11,Mexico,South Africa,J,home_win,0.702467,0.216343,0.081190,0.702467
1,2026-06-11,South Korea,Czech Republic,J,draw,0.264438,0.420343,0.315219,0.420343
2,2026-06-12,Canada,Bosnia and Herzegovina,D,home_win,0.600217,0.284234,0.115549,0.600217
3,2026-06-12,United States,Paraguay,B,away_win,0.182550,0.393533,0.423917,0.423917
4,2026-06-13,Qatar,Switzerland,D,away_win,0.043867,0.180518,0.775615,0.775615
5,2026-06-13,Brazil,Morocco,E,draw,0.267749,0.427371,0.304880,0.427371
6,2026-06-13,Haiti,Scotland,E,away_win,0.112536,0.343607,0.543857,0.543857
7,2026-06-13,Australia,Turkey,B,away_win,0.122320,0.360635,0.517045,0.517045
8,2026-06-14,Germany,Curaçao,I,home_win,0.718956,0.197609,0.083435,0.718956
9,2026-06-14,Ivory Coast,Ecuador,I,away_win,0.071320,0.273361,0.655319,0.655319


# 10. Save Predictions

In [50]:
X_schedule_stack = make_stacking_features(schedule_model)

schedule_probs = stack_model.predict_proba(X_schedule_stack)
schedule_preds = stack_model.predict(X_schedule_stack)

predictions_2026 = schedule_2026[
    [
        "date",
        "home_team",
        "away_team",
        "group",
        "is_placeholder_match",
        "has_market_odds",
    ]
].copy()

for i, class_name in enumerate(stack_model.classes_):
    predictions_2026[f"prob_{class_name}"] = schedule_probs[:, i]

predictions_2026["predicted_result"] = schedule_preds
predictions_2026["prediction_confidence"] = schedule_probs.max(axis=1)

def confidence_tier(x):
    if x >= 0.65:
        return "high"
    if x >= 0.50:
        return "medium"
    return "low"

predictions_2026["confidence_tier"] = predictions_2026["prediction_confidence"].apply(confidence_tier)

predictions_2026 = predictions_2026.sort_values("date").reset_index(drop=True)

winner_only = predictions_2026[
    [
        "date",
        "home_team",
        "away_team",
        "group",
        "predicted_result",
        "prediction_confidence",
        "confidence_tier",
        "prob_home_win",
        "prob_draw",
        "prob_away_win",
        "has_market_odds",
        "is_placeholder_match",
    ]
]

display(winner_only.head(40))

winner_only.to_csv("datasets/wc_2026_winner_predictions_only.csv", index=False)

,date,home_team,away_team,group,predicted_result,prediction_confidence,confidence_tier,prob_home_win,prob_draw,prob_away_win,has_market_odds,is_placeholder_match
0,2026-06-11,Mexico,South Africa,J,home_win,0.702467,high,0.702467,0.216343,0.081190,1,False
1,2026-06-11,South Korea,Czech Republic,J,draw,0.420343,low,0.264438,0.420343,0.315219,1,False
2,2026-06-12,Canada,Bosnia and Herzegovina,D,home_win,0.600217,medium,0.600217,0.284234,0.115549,1,False
3,2026-06-12,United States,Paraguay,B,away_win,0.423917,low,0.182550,0.393533,0.423917,1,False
4,2026-06-13,Qatar,Switzerland,D,away_win,0.775615,high,0.043867,0.180518,0.775615,1,False
5,2026-06-13,Brazil,Morocco,E,draw,0.427371,low,0.267749,0.427371,0.304880,1,False
6,2026-06-13,Haiti,Scotland,E,away_win,0.543857,medium,0.112536,0.343607,0.543857,1,False
7,2026-06-13,Australia,Turkey,B,away_win,0.517045,medium,0.122320,0.360635,0.517045,1,False
8,2026-06-14,Germany,Curaçao,I,home_win,0.718956,high,0.718956,0.197609,0.083435,1,False
9,2026-06-14,Ivory Coast,Ecuador,I,away_win,0.655319,high,0.071320,0.273361,0.655319,1,False
